# PPO notebook

#### Imports

In [210]:
from __future__ import annotations

import argparse
import json
import logging
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
import time
import gc
from copy import deepcopy


import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from tribes_env import TribesEnv

# Reduce Reuse Recycle!
from dqn import (
    JsonStateEncoder,
    DQNConfig,  # optional for shared defaults? prob not used
)

#### Config

In [ ]:
@dataclass
class PPOConfig:
    # Environment
    level_file: str = "tribes/levels/SampleLevel.csv"
    game_mode: str = "SCORE"
    seed: int = 123
    compile_first: bool = True

    # State encoder
    encoder_mode: str = "combined"
    engineered_dim: int = 1024
    raw_dim: int = 256

    # Network
    hidden_dim: int = 1024
    residual_blocks: int = 3

    # PPO
    learning_rate: float = 3e-4
    gamma: float = 0.99
    gae_lambda: float = 0.95

    clip_range: float = 0.2

    entropy_coef: float = 0.01
    value_coef: float = 0.5

    ppo_epochs: int = 4
    minibatch_size: int = 256 # unused as of now, probably a big improvement

    gradient_clip_norm: float = 1.0

    # Rollout collection
    rollout_steps: int = 512 # ?2048
    reward_scale: float = 100.0

    # Training schedule
    total_iterations: int = 1000

    # Evaluation
    eval_every_iterations: int = 25
    eval_episodes: int = 5
    eval_seed_offset: int = 10000
    eval_max_steps_per_episode: int = 512 # ?2048

    # Episode limits
    max_steps_per_episode: int = 2000

    # Checkpoints
    checkpoint_path: str = "checkpoints/ppo.pt"
    resume: bool = True

    # Device
    device: str = "mps" # cpu

#### Env Helper

In [212]:
def make_env(config: PPOConfig) -> TribesEnv:
    return TribesEnv(
        level_file=config.level_file,
        game_mode=config.game_mode,
        seed=config.seed,
        compile_first=False,
    )

encoder = JsonStateEncoder(
    mode="combined",
    engineered_dim=1024,
    raw_dim=256,
)

def evaluate_agent(
    agent,
    config,
    encoder,
    episodes,
    seed_offset=0,
):
    env = make_env(config)

    rewards = []

    t0 = time.time()

    print(f"[eval] start episodes={episodes}")

    for episode in range(episodes):

        ep_t0 = time.time()

        obs, info = env.reset(
            seed=config.seed + seed_offset + episode
        )

        state = encoder.encode(obs["state_json"])
        episode_reward = 0.0
        steps = 0

        for _ in range(config.eval_max_steps_per_episode):

            mask = np.asarray(info["action_mask"], dtype=np.bool_)

            action = agent.select_greedy_action(state, mask)

            next_obs, reward, terminated, truncated, next_info = env.step(action)

            episode_reward += reward
            steps += 1

            state = encoder.encode(next_obs["state_json"])
            info = next_info

            if terminated or truncated:
                break

        rewards.append(float(episode_reward))

        ep_time = time.time() - ep_t0

        print(
            f"[eval] episode={episode+1}/{episodes} "
            f"reward={episode_reward:.2f} "
            f"steps={steps} "
            f"time={ep_time:.2f}s"
        )

    env.close()

    mean_reward = float(np.mean(rewards))
    total_time = time.time() - t0

    print(
        f"[eval] done mean_reward={mean_reward:.3f} "
        f"total_time={total_time:.2f}s "
        f"avg_ep_time={total_time/episodes:.2f}s"
    )

    return {
        "episode_rewards": rewards,
        "mean_reward": mean_reward,
    }

#### Model definition

In [213]:
class ResidualBlock(nn.Module):
    def __init__(self, dim: int):
        super().__init__()

        self.fc1 = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)

        self.fc2 = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x

        x = self.fc1(x)
        x = self.norm1(x)
        x = F.gelu(x)

        x = self.fc2(x)
        x = self.norm2(x)

        x = x + residual

        return F.gelu(x)


class PPOModel(nn.Module):
    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        hidden_dim: int,
        residual_blocks: int,
    ):
        super().__init__()

        layers = [
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        ]

        for _ in range(residual_blocks):
            layers.append(
                ResidualBlock(hidden_dim)
            )

        self.trunk = nn.Sequential(*layers)

        self.policy_head = nn.Linear(
            hidden_dim,
            action_dim,
        )

        self.value_head = nn.Linear(
            hidden_dim,
            1,
        )

    def forward(self, states):
        features = self.trunk(states)

        logits = self.policy_head(features)

        values = self.value_head(
            features
        ).squeeze(-1)

        return logits, values

#### Rollout utilities

In [214]:
def masked_categorical(logits, action_masks,):
    logits = logits.masked_fill( ~action_masks, torch.finfo(logits.dtype).min,)
    return torch.distributions.Categorical(logits=logits)


def compute_gae(rewards, dones, values, last_value, gamma, gae_lambda,):
    advantages = []

    gae = 0.0

    values = values + [last_value]

    for t in reversed(
        range(len(rewards))
    ):
        delta = (
            rewards[t]
            + gamma
            * values[t + 1]
            * (1.0 - dones[t])
            - values[t]
        )

        gae = (
            delta
            + gamma
            * gae_lambda
            * (1.0 - dones[t])
            * gae
        )

        advantages.insert(0, gae)

    returns = [
        adv + value
        for adv, value
        in zip(
            advantages,
            values[:-1],
        )
    ]

    return advantages, returns

@dataclass
class RolloutBuffer:
    states: list
    actions: list
    rewards: list
    dones: list

    values: list
    log_probs: list

    masks: list

    final_state: np.ndarray
    final_mask: np.ndarray

#### PPO Agent Definition

In [215]:
class PPOAgent:
    def __init__(
        self,
        state_dim,
        action_dim,
        config,
    ):
        self.config = config

        self.device = torch.device(
            config.device
        )

        self.model = PPOModel(
            state_dim,
            action_dim,
            config.hidden_dim,
            config.residual_blocks,
        ).to(self.device)

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=config.learning_rate,
        )

    @torch.no_grad()
    def select_action(
        self,
        state,
        action_mask,
    ):
        state_tensor = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        mask_tensor = torch.as_tensor(
            action_mask,
            dtype=torch.bool,
            device=self.device,
        ).unsqueeze(0)

        logits, values = self.model(
            state_tensor
        )

        dist = masked_categorical(
            logits,
            mask_tensor,
        )

        action = dist.sample()

        return (
            int(action.item()),
            float(dist.log_prob(action).item()),
            float(values.item()),
        )
    
    @torch.no_grad()
    def select_greedy_action(
        self,
        state,
        action_mask,
    ):
        state_tensor = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        mask_tensor = torch.as_tensor(
            action_mask,
            dtype=torch.bool,
            device=self.device,
        ).unsqueeze(0)

        logits, _ = self.model(state_tensor)

        logits = logits.masked_fill(
            ~mask_tensor,
            torch.finfo(logits.dtype).min,
        )

        action = torch.argmax(
            logits,
            dim=1,
        )

        return int(action.item())
    
    def predict_value(self,state):
        state_tensor = torch.as_tensor(state, dtype=torch.float32, device=self.device,).unsqueeze(0)

        _, value = self.model(state_tensor)

        return float(value.squeeze(0).item())

#### PPO Update loop

In [216]:
def ppo_update(
    self,
    states,
    actions,
    old_log_probs,
    returns,
    advantages,
    action_masks,
):
    logits, values = self.model(
        states
    )

    dist = masked_categorical(
        logits,
        action_masks,
    )

    new_log_probs = dist.log_prob(
        actions
    )

    entropy = dist.entropy()

    ratio = torch.exp(
        new_log_probs
        - old_log_probs
    )

    surr1 = ratio * advantages

    surr2 = (
        torch.clamp(
            ratio,
            1.0 - self.config.clip_range,
            1.0 + self.config.clip_range,
        )
        * advantages
    )

    policy_loss = (
        -torch.min(
            surr1,
            surr2,
        ).mean()
    )

    value_loss = F.mse_loss(
        values,
        returns,
    )

    entropy_loss = entropy.mean()

    loss = (
        policy_loss
        + self.config.value_coef
        * value_loss
        - self.config.entropy_coef
        * entropy_loss
    )

    self.optimizer.zero_grad(
        set_to_none=True
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        self.model.parameters(),
        self.config.gradient_clip_norm,
    )

    self.optimizer.step()

    return {
        "loss": float(loss.item()),
        "policy_loss": float(
            policy_loss.item()
        ),
        "value_loss": float(
            value_loss.item()
        ),
        "entropy": float(
            entropy_loss.item()
        ),
    }


PPOAgent.update = ppo_update

#### Rollout Collection

In [217]:
def collect_rollout(
    env,
    agent,
    encoder,
    config,
):
    obs, info = env.reset(
        seed=config.seed
    )

    state = encoder.encode(
        obs["state_json"]
    )

    rollout = RolloutBuffer(
        states=[],
        actions=[],
        rewards=[],
        dones=[],
        values=[],
        log_probs=[],
        masks=[],
        final_state = None,
        final_mask = None
    )

    while (len(rollout.states) < config.rollout_steps):
        mask = np.asarray(
            info["action_mask"],
            dtype=np.bool_,
        )

        (action, log_prob, value,) = agent.select_action(state,mask,)

        (next_obs,reward,terminated,truncated,next_info,) = env.step(action)

        rollout.states.append(state)
        rollout.actions.append(action)
        rollout.rewards.append(reward / config.reward_scale)
        rollout.dones.append(terminated or truncated)
        rollout.values.append(value)
        rollout.log_probs.append(log_prob)
        rollout.masks.append(mask)
        
        state = encoder.encode(next_obs["state_json"])

        info = next_info

        if terminated or truncated:
            obs, info = env.reset()
            state = encoder.encode(obs["state_json"])
    
    rollout.final_state = state
    rollout.final_mask = np.asarray(
        info["action_mask"],
        dtype=np.bool_,
    )

    return rollout

#### Train Loop

In [218]:
# prep env and checkpoint pathing
config = PPOConfig()

Path(config.checkpoint_path).parent.mkdir(parents=True, exist_ok=True)

env = make_env(config)
obs, info = env.reset()
state_dim = encoder.encode(obs["state_json"]).shape[0]
action_dim = env.action_space.n

best_eval_reward = float("-inf")
start_iteration = 0

base = Path(config.checkpoint_path)
last_path = base.parent / f"{base.stem}_last.pt"
best_path = base.parent / f"{base.stem}_best.pt"

if config.resume and last_path.exists():
    print(f"Loading checkpoint: {last_path}")
    checkpoint = torch.load(last_path, map_location=config.device)

    agent = PPOAgent(state_dim, action_dim, config)

    agent.model.load_state_dict(checkpoint["model_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_iteration = checkpoint.get("iteration", 0)
    best_eval_reward = checkpoint.get("best_eval_reward", float("-inf"))

else:
    agent = PPOAgent(state_dim, action_dim, config)

checkpoint = {}

Loading checkpoint: checkpoints/ppo_last.pt


In [219]:
# actual train loop
for iteration in range(start_iteration, config.total_iterations):
    iter_time = time.time()
    rollout = collect_rollout(
        env,
        agent,
        encoder,
        config,
    )

    # predict value if reaching end of rollout
    last_state = rollout.final_state
    last_mask = rollout.final_mask
    last_value = agent.predict_value(last_state)

    advantages, returns = compute_gae(
        rollout.rewards,
        rollout.dones,
        rollout.values,
        last_value=last_value,
        gamma=config.gamma,
        gae_lambda=config.gae_lambda,
    )

    # as_tensor doesn't copy input data, just reuses existing space
    states = torch.as_tensor(
        np.array(rollout.states),
        dtype=torch.float32,
        device=agent.device,
    )

    actions = torch.as_tensor(
        rollout.actions,
        dtype=torch.long,
        device=agent.device,
    )

    old_log_probs = torch.as_tensor(
        rollout.log_probs,
        dtype=torch.float32,
        device=agent.device,
    )

    returns_t = torch.as_tensor(
        returns,
        dtype=torch.float32,
        device=agent.device,
    )

    advantages_t = torch.as_tensor(
        advantages,
        dtype=torch.float32,
        device=agent.device,
    )

    advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

    action_masks = torch.as_tensor(
        np.array(rollout.masks),
        dtype=torch.bool,
        device=agent.device,
    )

    for _ in range(config.ppo_epochs):
        stats = agent.update(
            states,
            actions,
            old_log_probs,
            returns_t,
            advantages_t,
            action_masks,
        )

    # separate iter train time from eval
    iter_time = time.time() - iter_time

    # evaluate agent
    eval_reward = None

    if (config.eval_every_iterations > 0
        and (iteration + 1) % config.eval_every_iterations == 0):
        eval_results = evaluate_agent(
            agent,
            config,
            encoder,
            config.eval_episodes,
            seed_offset=(config.eval_seed_offset + iteration),
        )

        eval_reward = (eval_results["mean_reward"])

        # latest checkpointing
        checkpoint = {
            "iteration": iteration + 1,
            "model_state_dict": agent.model.state_dict(),
            "optimizer_state_dict": agent.optimizer.state_dict(),
            "config": asdict(config),
            "best_eval_reward": best_eval_reward,
        }

        print("Saving latest checkpoint")
        torch.save(checkpoint, last_path)

        if eval_reward > best_eval_reward:
            print("Saving best checkpoint")
            best_eval_reward = eval_reward
            checkpoint["best_eval_reward"] = best_eval_reward
            torch.save(checkpoint, best_path)

    print(
        f"iter={iteration + 1}",
        f"loss={stats['loss']:+.4f}",
        f"policy={stats['policy_loss']:+.4f}",
        f"value={stats['value_loss']:+.4f}",
        f"entropy={stats['entropy']:+.4f}",
        f"time (iter)={iter_time:.1f}s"
    )

    # drop references to rollout tensors/lists
    del rollout
    del states, actions, old_log_probs, returns_t, advantages_t, action_masks

    # try and garbage collect
    gc.collect() 
    if agent.device.type == "cuda":
        torch.cuda.empty_cache()

iter=31 loss=+0.0880 policy=-0.0106 value=+0.2263 entropy=+1.4468 time (iter)=7.3s
iter=32 loss=+0.0611 policy=-0.0087 value=+0.1699 entropy=+1.5143 time (iter)=6.9s
iter=33 loss=+0.0544 policy=-0.0093 value=+0.1554 entropy=+1.4019 time (iter)=8.0s
iter=34 loss=+0.1745 policy=-0.0007 value=+0.3788 entropy=+1.4234 time (iter)=7.1s
iter=35 loss=+0.0565 policy=-0.0009 value=+0.1419 entropy=+1.3579 time (iter)=7.0s
iter=36 loss=+0.0392 policy=+0.0045 value=+0.0953 entropy=+1.2897 time (iter)=7.1s
iter=37 loss=+0.0998 policy=+0.0432 value=+0.1392 entropy=+1.3019 time (iter)=7.2s
iter=38 loss=+0.0655 policy=-0.0083 value=+0.1718 entropy=+1.2107 time (iter)=7.4s
iter=39 loss=+0.0242 policy=-0.0044 value=+0.0793 entropy=+1.1037 time (iter)=6.8s
iter=40 loss=+0.0462 policy=+0.0099 value=+0.0947 entropy=+1.1081 time (iter)=7.1s
iter=41 loss=+0.0290 policy=-0.0134 value=+0.1079 entropy=+1.1604 time (iter)=7.5s
iter=42 loss=+0.1330 policy=-0.0023 value=+0.2944 entropy=+1.1957 time (iter)=7.1s
iter

KeyboardInterrupt: 